# Análisis de tiempos de pago

**Proyecto final de Ciencia de Datos DataXperience**  
**Integrantes:** Sergio David Morales Forero      

Queremos mirar si la cantidad de operaciones de un lote tiene relación con el tiempo que tarda. Usamos **datos simulados**, no información real de un banco.

## 1. Cargar y revisar los datos

Subimos el archivo `pagos_simulados.csv` que está en la carpeta `datos` del proyecto. Cada fila representa un lote de operaciones.

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
from google.colab import files
files.upload()
df = pd.read_csv('pagos_simulados.csv')
df.head()

In [ ]:
print(df.shape)
print(df.dtypes)
print(df.isna().sum())
print('Duplicados:', df.duplicated().sum())

## 2. Limpiar los datos

Eliminamos los registros repetidos, los que tienen campos vacíos y los que tienen un tiempo igual a cero. También dejamos escritos los tipos de proceso de la misma forma.

In [ ]:
df = df.drop_duplicates()
df = df.dropna()
df = df[df['minutos'] > 0].copy()
df['tipo'] = df['tipo'].str.strip().str.title().replace({'Nomina': 'Nómina'})
print('Registros finales:', len(df))
df.head()

## 3. Estadística descriptiva

Revisamos cuánto duran los lotes en promedio y cuánto cambian los tiempos entre unos y otros.

In [ ]:
print('Media:', round(df['minutos'].mean(), 1))
print('Mediana:', round(df['minutos'].median(), 1))
print('Moda:', df['minutos'].mode()[0])
print('Rango:', round(df['minutos'].max() - df['minutos'].min(), 1))
print('Varianza:', round(df['minutos'].var(), 1))
print('Desviación:', round(df['minutos'].std(), 1))
print(df.groupby('tipo')['minutos'].mean().round(1))

In [ ]:
df['minutos'].hist(bins=10)
plt.title('Duración de los lotes')
plt.xlabel('Minutos')
plt.ylabel('Cantidad de lotes')
plt.show()

In [ ]:
df.groupby('tipo')['minutos'].mean().plot(kind='bar')
plt.title('Tiempo promedio por proceso')
plt.xlabel('Proceso')
plt.ylabel('Minutos')
plt.xticks(rotation=0)
plt.show()

In [ ]:
plt.scatter(df['operaciones'], df['minutos'])
plt.title('Operaciones y tiempo')
plt.xlabel('Número de operaciones')
plt.ylabel('Minutos')
plt.show()

**Lo que observamos:** después de limpiar quedaron **120 lotes**. El tiempo promedio fue de **63.8 minutos**. En el gráfico de puntos se ve que, en general, cuando hay más operaciones el tiempo aumenta. Hay diferencias entre procesos y no todos los puntos siguen exactamente la misma tendencia.

## 4. Modelo sencillo

Usamos regresión lineal para estimar el tiempo a partir de la cantidad de operaciones. Reservamos una parte de los datos para comprobar qué tan cerca quedan las estimaciones.

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_absolute_error

X = df[['operaciones']]
y = df['minutos']
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=1)
modelo = LinearRegression()
modelo.fit(X_train, y_train)
predicciones = modelo.predict(X_test)
print('Error promedio:', round(mean_absolute_error(y_test, predicciones), 1), 'minutos')
print('Tiempo para 100 operaciones:', round(modelo.predict(pd.DataFrame({'operaciones': [100]}))[0], 1), 'minutos')

## 5. Conclusión

En esta base de práctica, los lotes con más operaciones suelen tardar más. El modelo tuvo un error promedio de **7.8 minutos** en los datos de prueba, así que sirve para aprender, pero no es exacto.

Desde Ingeniería Industrial, esta idea podría servir para organizar mejor el trabajo. Para aplicarla en una empresa sería necesario estudiar datos reales con autorización y tener en cuenta otros factores, como el personal disponible o las fallas de los sistemas.

**Importante:** todos los datos de este ejercicio son simulados.